In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from sklearn.model_selection import StratifiedKFold
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

COMBINED_DIR = Path("/kaggle/input/datasets/aondonamoses/abreastultrasound/Ultrasound/combined_dataset")
AFRICAN_DIR = Path("/kaggle/input/datasets/aondonamoses/abreastultrasound/Ultrasound/african_validation")

IMAGE_SIZE = 256
BATCH_SIZE = 16
NUM_WORKERS = 2
NUM_FOLDS = 5
SEED = 42


def acoustic_shadowing_aug(image, **kwargs):
    h, w = image.shape[:2]
    shadow = np.ones((h, w), dtype=np.float32)
    num_shadows = np.random.randint(1, 3)
    for _ in range(num_shadows):
        x = np.random.randint(0, w)
        width = np.random.randint(w // 10, w // 4)
        beta = np.random.uniform(0.3, 0.6)
        x1 = max(0, x - width // 2)
        x2 = min(w, x + width // 2)
        depth_start = np.random.randint(h // 4, h // 2)
        for row in range(depth_start, h):
            attenuation = beta * ((row - depth_start) / (h - depth_start))
            shadow[row, x1:x2] *= (1.0 - attenuation)
    image = (image * shadow[:, :, np.newaxis]).clip(0, 255).astype(np.uint8)
    return image


def gain_variation_aug(image, **kwargs):
    h = image.shape[0]
    gain = np.linspace(
        np.random.uniform(0.7, 1.0),
        np.random.uniform(1.0, 1.3),
        h
    ).astype(np.float32)
    image = (image * gain[:, np.newaxis, np.newaxis]).clip(0, 255).astype(np.uint8)
    return image


def get_train_transforms():
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=15, p=0.5),
        A.RandomScale(scale_limit=0.1, p=0.5),
        A.PadIfNeeded(min_height=IMAGE_SIZE, min_width=IMAGE_SIZE, border_mode=0, p=1.0),
        A.CenterCrop(height=IMAGE_SIZE, width=IMAGE_SIZE, p=1.0),
        A.RandomBrightnessContrast(p=0.5),
        A.OneOf([
            A.GaussNoise(std_range=(0.05, 0.2), p=1.0),
            A.MultiplicativeNoise(multiplier=(0.8, 1.2), p=1.0),
        ], p=0.5),
        A.Lambda(image=acoustic_shadowing_aug, p=0.5),
        A.Lambda(image=gain_variation_aug, p=0.5),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ], additional_targets={"mask": "mask"})


def get_val_transforms():
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ], additional_targets={"mask": "mask"})


def get_tta_transforms():
    base = [
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ]

    def make(augments):
        return A.Compose(
            [A.Resize(IMAGE_SIZE, IMAGE_SIZE)] + augments + [
                A.PadIfNeeded(min_height=IMAGE_SIZE, min_width=IMAGE_SIZE, border_mode=0, p=1.0),
                A.CenterCrop(height=IMAGE_SIZE, width=IMAGE_SIZE, p=1.0),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
                ToTensorV2(),
            ],
            additional_targets={"mask": "mask"}
        )

    return [
        make([A.HorizontalFlip(p=1.0)]),
        make([A.Rotate(limit=(90, 90), p=1.0)]),
        make([A.Rotate(limit=(180, 180), p=1.0)]),
        make([A.Rotate(limit=(270, 270), p=1.0)]),
        make([A.RandomScale(scale_limit=(-0.1, -0.1), p=1.0)]),
        make([A.RandomScale(scale_limit=(0.1, 0.1), p=1.0)]),
        A.Compose(base, additional_targets={"mask": "mask"}),
    ]


class BreastUltrasoundDataset(Dataset):
    def __init__(self, dataframe, base_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.base_dir = Path(base_dir)
        self.transform = transform
        self.label_map = {"benign": 0, "malignant": 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = np.array(Image.open(self.base_dir / row["image"]).convert("RGB"))
        mask = np.array(Image.open(self.base_dir / row["mask"]).convert("L"))
        mask = (mask > 127).astype(np.float32)
        label = self.label_map[row["label"]]

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"].unsqueeze(0)
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask, torch.tensor(label, dtype=torch.long)


class AfricanValidationDataset(Dataset):
    def __init__(self, dataframe, base_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.base_dir = Path(base_dir)
        self.transform = transform
        self.label_map = {"benign": 0, "malignant": 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = np.array(Image.open(self.base_dir / row["image"]).convert("RGB"))
        mask = np.array(Image.open(self.base_dir / row["mask"]).convert("L"))
        mask = (mask > 127).astype(np.float32)
        label = self.label_map[row["label"]]

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"].unsqueeze(0)
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask, torch.tensor(label, dtype=torch.long)


def get_cv_folds(manifest_path, n_splits=NUM_FOLDS, seed=SEED):
    df = pd.read_csv(manifest_path)
    df["label_int"] = (df["label"] == "malignant").astype(int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = []
    for train_idx, val_idx in skf.split(df, df["label_int"]):
        folds.append((df.iloc[train_idx].copy(), df.iloc[val_idx].copy()))
    return folds


def get_dataloaders(train_df, val_df, base_dir, use_us_aug=True):
    train_transform = get_train_transforms() if use_us_aug else get_val_transforms()
    val_transform = get_val_transforms()

    train_dataset = BreastUltrasoundDataset(train_df, base_dir, transform=train_transform)
    val_dataset = BreastUltrasoundDataset(val_df, base_dir, transform=val_transform)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )
    return train_loader, val_loader


def get_african_loader(transform=None):
    df = pd.read_csv(AFRICAN_DIR / "manifest.csv")
    if transform is None:
        transform = get_val_transforms()
    dataset = AfricanValidationDataset(df, AFRICAN_DIR, transform=transform)
    loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )
    return loader


def get_african_loov_splits():
    df = pd.read_csv(AFRICAN_DIR / "manifest.csv")
    splits = []
    for i in range(len(df)):
        test_df = df.iloc[[i]].copy()
        train_df = df.drop(index=i).reset_index(drop=True).copy()
        splits.append((train_df, test_df))
    return splits


if __name__ == "__main__":
    folds = get_cv_folds(COMBINED_DIR / "manifest.csv")
    print(f"Number of folds: {len(folds)}")
    for i, (train_df, val_df) in enumerate(folds):
        print(f"Fold {i}: train={len(train_df)}, val={len(val_df)}, "
              f"train_malignant={train_df['label'].eq('malignant').sum()}, "
              f"val_malignant={val_df['label'].eq('malignant').sum()}")

    train_loader, val_loader = get_dataloaders(folds[0][0], folds[0][1], COMBINED_DIR)
    images, masks, labels = next(iter(train_loader))
    print(f"\nBatch check:")
    print(f"  images: {images.shape}")
    print(f"  masks:  {masks.shape}")
    print(f"  labels: {labels.shape}, unique: {labels.unique()}")

    african_loader = get_african_loader()
    images, masks, labels = next(iter(african_loader))
    print(f"\nAfrican batch check:")
    print(f"  images: {images.shape}")
    print(f"  masks:  {masks.shape}")
    print(f"  labels: {labels}")

    loov_splits = get_african_loov_splits()
    print(f"\nAfrican LOOV splits: {len(loov_splits)}")

Number of folds: 5
Fold 0: train=2218, val=555, train_malignant=731, val_malignant=183
Fold 1: train=2218, val=555, train_malignant=731, val_malignant=183
Fold 2: train=2218, val=555, train_malignant=731, val_malignant=183
Fold 3: train=2219, val=554, train_malignant=732, val_malignant=182
Fold 4: train=2219, val=554, train_malignant=731, val_malignant=183


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Batch check:
  images: torch.Size([16, 3, 256, 256])
  masks:  torch.Size([16, 1, 256, 256])
  labels: torch.Size([16]), unique: tensor([0, 1])


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



African batch check:
  images: torch.Size([1, 3, 256, 256])
  masks:  torch.Size([1, 1, 256, 256])
  labels: tensor([0])

African LOOV splits: 20


In [3]:
!pip install segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 1.1 MB/s eta 0:00:00a 0:00:01


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp


ENCODER_CONFIGS = {
    "resnet34": {
        "encoder_name": "resnet34",
        "encoder_weights": "imagenet",
        "encoder_depth": 5,
    },
    "resnet50": {
        "encoder_name": "resnet50",
        "encoder_weights": "imagenet",
        "encoder_depth": 5,
    },
    "efficientnet-b3": {
        "encoder_name": "efficientnet-b3",
        "encoder_weights": "imagenet",
        "encoder_depth": 5,
    },
}


class ClassificationHead(nn.Module):
    def __init__(self, in_channels, num_classes=2, dropout_p=0.3):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout1 = nn.Dropout(p=dropout_p)
        self.fc1 = nn.Linear(in_channels, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.dropout2 = nn.Dropout(p=dropout_p)
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.dropout3 = nn.Dropout(p=dropout_p)
        self.fc3 = nn.Linear(256, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.dropout4 = nn.Dropout(p=dropout_p)
        self.fc4 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(x).flatten(1)
        x = self.dropout1(x)
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout2(x)
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout3(x)
        x = F.relu(self.bn3(self.fc3(x)))
        x = self.dropout4(x)
        return self.fc4(x)


class DualTaskModel(nn.Module):
    def __init__(self, encoder_name="efficientnet-b3", dropout_p=0.3, num_classes=2):
        super().__init__()

        if encoder_name not in ENCODER_CONFIGS:
            raise ValueError(f"encoder_name must be one of {list(ENCODER_CONFIGS.keys())}")

        config = ENCODER_CONFIGS[encoder_name]

        self.segmentation_model = smp.Unet(
            encoder_name=config["encoder_name"],
            encoder_weights=config["encoder_weights"],
            encoder_depth=config["encoder_depth"],
            decoder_channels=(256, 128, 64, 32, 16),
            in_channels=3,
            classes=1,
            activation=None,
        )

        encoder_out_channels = self.segmentation_model.encoder.out_channels[-1]

        self.classification_head = ClassificationHead(
            in_channels=encoder_out_channels,
            num_classes=num_classes,
            dropout_p=dropout_p,
        )

    def forward(self, x):
        features = self.segmentation_model.encoder(x)
        decoder_output = self.segmentation_model.decoder(features)
        seg_logits = self.segmentation_model.segmentation_head(decoder_output)
        cls_logits = self.classification_head(features[-1])
        return seg_logits, cls_logits

    def enable_dropout(self):
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.train()

    def predict_with_uncertainty(self, x, n_passes=20):
        self.eval()
        self.enable_dropout()

        seg_preds = []
        cls_preds = []

        with torch.no_grad():
            for _ in range(n_passes):
                seg_logits, cls_logits = self.forward(x)
                seg_preds.append(torch.sigmoid(seg_logits))
                cls_preds.append(torch.softmax(cls_logits, dim=1))

        seg_preds = torch.stack(seg_preds, dim=0)
        cls_preds = torch.stack(cls_preds, dim=0)

        seg_mean = seg_preds.mean(dim=0)
        seg_uncertainty = seg_preds.var(dim=0)

        cls_mean = cls_preds.mean(dim=0)
        cls_uncertainty = cls_preds.var(dim=0).sum(dim=1)

        cls_label = cls_mean.argmax(dim=1)
        cls_confidence = cls_mean.max(dim=1).values

        return {
            "seg_mean": seg_mean,
            "seg_uncertainty": seg_uncertainty,
            "cls_mean": cls_mean,
            "cls_label": cls_label,
            "cls_confidence": cls_confidence,
            "cls_uncertainty": cls_uncertainty,
        }


def get_model(encoder_name="efficientnet-b3", dropout_p=0.3):
    return DualTaskModel(encoder_name=encoder_name, dropout_p=dropout_p)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)



In [5]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    for encoder in ENCODER_CONFIGS.keys():
        model = get_model(encoder_name=encoder).to(device)
        params = count_parameters(model)
        print(f"\n{encoder}: {params:,} parameters")

        x = torch.randn(2, 3, 256, 256).to(device)
        seg_logits, cls_logits = model(x)
        print(f"  seg_logits: {seg_logits.shape}")
        print(f"  cls_logits: {cls_logits.shape}")

        out = model.predict_with_uncertainty(x, n_passes=20)
        print(f"  seg_mean:        {out['seg_mean'].shape}")
        print(f"  seg_uncertainty: {out['seg_uncertainty'].shape}")
        print(f"  cls_label:       {out['cls_label']}")
        print(f"  cls_confidence:  {out['cls_confidence']}")
        print(f"  cls_uncertainty: {out['cls_uncertainty']}")

Device: cpu


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]


resnet34: 24,865,299 parameters
  seg_logits: torch.Size([2, 1, 256, 256])
  cls_logits: torch.Size([2, 2])
  seg_mean:        torch.Size([2, 1, 256, 256])
  seg_uncertainty: torch.Size([2, 1, 256, 256])
  cls_label:       tensor([1, 1])
  cls_confidence:  tensor([0.5333, 0.5281])
  cls_uncertainty: tensor([0.0004, 0.0012])


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]


resnet50: 33,736,467 parameters
  seg_logits: torch.Size([2, 1, 256, 256])
  cls_logits: torch.Size([2, 2])
  seg_mean:        torch.Size([2, 1, 256, 256])
  seg_uncertainty: torch.Size([2, 1, 256, 256])
  cls_label:       tensor([0, 0])
  cls_confidence:  tensor([0.5272, 0.5234])
  cls_uncertainty: tensor([0.0001, 0.0003])


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]


efficientnet-b3: 13,522,427 parameters
  seg_logits: torch.Size([2, 1, 256, 256])
  cls_logits: torch.Size([2, 2])
  seg_mean:        torch.Size([2, 1, 256, 256])
  seg_uncertainty: torch.Size([2, 1, 256, 256])
  cls_label:       tensor([1, 0])
  cls_confidence:  tensor([0.5157, 0.5017])
  cls_uncertainty: tensor([0.0014, 0.0028])


In [6]:
import os
import time
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LR = 1e-4
WEIGHT_DECAY = 1e-5
MAX_EPOCHS = 100
PATIENCE = 15
SEG_WEIGHT = 0.6
CLS_WEIGHT = 0.4

CONFIGURATIONS = [
    {"name": "resnet50_baseline",      "encoder": "resnet50",        "us_aug": False, "tta": False},
    {"name": "resnet50_tta",           "encoder": "resnet50",        "us_aug": False, "tta": True},
    {"name": "resnet50_usaug",         "encoder": "resnet50",        "us_aug": True,  "tta": False},
    {"name": "resnet34_baseline",      "encoder": "resnet34",        "us_aug": False, "tta": False},
    {"name": "efficientnet_baseline",  "encoder": "efficientnet-b3", "us_aug": False, "tta": False},
    {"name": "efficientnet_tta",       "encoder": "efficientnet-b3", "us_aug": False, "tta": True},
    {"name": "efficientnet_usaug",     "encoder": "efficientnet-b3", "us_aug": True,  "tta": False},
    {"name": "efficientnet_usaug_tta", "encoder": "efficientnet-b3", "us_aug": True,  "tta": True},
]


def dice_loss(pred, target, smooth=1.0):
    pred = torch.sigmoid(pred)
    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)
    intersection = (pred * target).sum()
    return 1 - (2.0 * intersection + smooth) / (pred.sum() + target.sum() + smooth)


def seg_loss_fn(pred, target):
    return 0.5 * dice_loss(pred, target) + 0.5 * F.binary_cross_entropy_with_logits(pred, target)


def compute_metrics(seg_preds, seg_targets, cls_probs, cls_targets):
    seg_preds_bin = (seg_preds > 0.5).float()
    intersection = (seg_preds_bin * seg_targets).sum()
    dice = (2.0 * intersection + 1.0) / (seg_preds_bin.sum() + seg_targets.sum() + 1.0)
    union = seg_preds_bin.sum() + seg_targets.sum() - intersection
    iou = (intersection + 1.0) / (union + 1.0)

    cls_np = cls_targets.cpu().numpy()
    cls_prob_np = cls_probs.cpu().numpy()
    cls_pred_np = cls_prob_np.argmax(axis=1)

    accuracy = accuracy_score(cls_np, cls_pred_np)
    precision = precision_score(cls_np, cls_pred_np, zero_division=0)
    recall = recall_score(cls_np, cls_pred_np, zero_division=0)
    f1 = f1_score(cls_np, cls_pred_np, zero_division=0)
    try:
        auc = roc_auc_score(cls_np, cls_prob_np[:, 1])
    except ValueError:
        auc = 0.0

    return {
        "dice": dice.item(),
        "iou": iou.item(),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
    }


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for images, masks, labels in loader:
        images = images.to(device)
        masks = masks.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        seg_logits, cls_logits = model(images)
        loss = SEG_WEIGHT * seg_loss_fn(seg_logits, masks) + CLS_WEIGHT * F.cross_entropy(cls_logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, device, use_tta=False, tta_transforms=None):
    model.eval()
    total_loss = 0.0
    all_seg_preds = []
    all_seg_targets = []
    all_cls_probs = []
    all_cls_targets = []

    for images, masks, labels in loader:
        images = images.to(device)
        masks = masks.to(device)
        labels = labels.to(device)

        if use_tta and tta_transforms is not None:
            seg_tta = []
            cls_tta = []
            for tta_tf in tta_transforms:
                images_np = images.cpu().numpy().transpose(0, 2, 3, 1)
                images_np = ((images_np * np.array([0.229, 0.224, 0.225])) +
                             np.array([0.485, 0.456, 0.406])) * 255
                images_np = images_np.clip(0, 255).astype(np.uint8)
                aug_images = []
                for img in images_np:
                    aug = tta_tf(image=img, mask=np.zeros(img.shape[:2], dtype=np.float32))
                    aug_images.append(aug["image"])
                aug_tensor = torch.stack(aug_images).to(device)
                s, c = model(aug_tensor)
                seg_tta.append(torch.sigmoid(s))
                cls_tta.append(torch.softmax(c, dim=1))
            seg_pred = torch.stack(seg_tta).mean(dim=0)
            cls_prob = torch.stack(cls_tta).mean(dim=0)
            seg_logits, cls_logits = model(images)
            loss = (SEG_WEIGHT * seg_loss_fn(seg_logits, masks) +
                    CLS_WEIGHT * F.cross_entropy(cls_logits, labels))
        else:
            seg_logits, cls_logits = model(images)
            loss = (SEG_WEIGHT * seg_loss_fn(seg_logits, masks) +
                    CLS_WEIGHT * F.cross_entropy(cls_logits, labels))
            seg_pred = torch.sigmoid(seg_logits)
            cls_prob = torch.softmax(cls_logits, dim=1)

        total_loss += loss.item()
        all_seg_preds.append(seg_pred.cpu())
        all_seg_targets.append(masks.cpu())
        all_cls_probs.append(cls_prob.cpu())
        all_cls_targets.append(labels.cpu())

    all_seg_preds = torch.cat(all_seg_preds)
    all_seg_targets = torch.cat(all_seg_targets)
    all_cls_probs = torch.cat(all_cls_probs)
    all_cls_targets = torch.cat(all_cls_targets)

    metrics = compute_metrics(all_seg_preds, all_seg_targets, all_cls_probs, all_cls_targets)
    metrics["loss"] = total_loss / len(loader)
    return metrics


def train_configuration(config, folds, device):
    print(f"\n{'='*60}")
    print(f"Configuration: {config['name']}")
    print(f"{'='*60}")

    tta_transforms = get_tta_transforms() if config["tta"] else None
    fold_results = []

    for fold_idx, (train_df, val_df) in enumerate(folds):
        print(f"\nFold {fold_idx + 1}/{NUM_FOLDS}")

        train_loader, val_loader = get_dataloaders(
            train_df, val_df, COMBINED_DIR, use_us_aug=config["us_aug"]
        )

        model = get_model(encoder_name=config["encoder"]).to(device)
        optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)

        best_dice = 0.0
        patience_counter = 0
        best_metrics = None
        best_epoch = 0

        for epoch in range(MAX_EPOCHS):
            t0 = time.time()
            train_loss = train_one_epoch(model, train_loader, optimizer, device)
            val_metrics = validate(model, val_loader, device,
                                   use_tta=config["tta"], tta_transforms=tta_transforms)
            scheduler.step()
            elapsed = time.time() - t0

            print(f"  Epoch {epoch+1:03d} | loss={train_loss:.4f} | "
                  f"dice={val_metrics['dice']:.4f} | auc={val_metrics['auc']:.4f} | "
                  f"recall={val_metrics['recall']:.4f} | {elapsed:.1f}s")

            if val_metrics["dice"] > best_dice:
                best_dice = val_metrics["dice"]
                best_metrics = val_metrics
                best_epoch = epoch + 1
                patience_counter = 0
                ckpt_path = CHECKPOINT_DIR / f"{config['name']}_fold{fold_idx}.pth"
                torch.save(model.state_dict(), ckpt_path)
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    print(f"  Early stopping at epoch {epoch+1}")
                    break

        print(f"  Best epoch: {best_epoch} | dice={best_metrics['dice']:.4f} | "
              f"auc={best_metrics['auc']:.4f} | recall={best_metrics['recall']:.4f} | "
              f"accuracy={best_metrics['accuracy']:.4f}")

        fold_results.append(best_metrics)

    summary = {}
    for key in fold_results[0].keys():
        vals = [r[key] for r in fold_results]
        summary[key] = {"mean": np.mean(vals), "std": np.std(vals)}

    print(f"\nSummary: {config['name']}")
    for key, val in summary.items():
        print(f"  {key}: {val['mean']:.4f} +/- {val['std']:.4f}")

    return fold_results, summary


def run_all_configurations():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    folds = get_cv_folds(COMBINED_DIR / "manifest.csv")
    all_results = {}

    for config in CONFIGURATIONS:
        fold_results, summary = train_configuration(config, folds, device)
        all_results[config["name"]] = {"fold_results": fold_results, "summary": summary}

    rows = []
    for config_name, result in all_results.items():
        row = {"configuration": config_name}
        for metric, vals in result["summary"].items():
            row[f"{metric}_mean"] = round(vals["mean"], 4)
            row[f"{metric}_std"] = round(vals["std"], 4)
        rows.append(row)

    results_df = pd.DataFrame(rows)
    results_df.to_csv("/kaggle/working/all_results.csv", index=False)
    print("\nResults saved to /kaggle/working/all_results.csv")
    return all_results

In [7]:
def get_completed_configurations():
    completed = []
    for config in CONFIGURATIONS:
        path = Path(f"/kaggle/working/{config['name']}_results.csv")
        if path.exists():
            completed.append(config["name"])
    return completed

def run_remaining_configurations():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    completed = get_completed_configurations()
    print(f"Already completed: {completed}")

    folds = get_cv_folds(COMBINED_DIR / "manifest.csv")
    all_results = {}

    for config in CONFIGURATIONS:
        if config["name"] in completed:
            print(f"Skipping {config['name']} (already done)")
            continue

        fold_results, summary = train_configuration(config, folds, device)
        all_results[config["name"]] = {"fold_results": fold_results, "summary": summary}

        config_row = {"configuration": config["name"]}
        for metric, vals in summary.items():
            config_row[f"{metric}_mean"] = round(vals["mean"], 4)
            config_row[f"{metric}_std"] = round(vals["std"], 4)
        pd.DataFrame([config_row]).to_csv(
            f"/kaggle/working/{config['name']}_results.csv", index=False
        )

    rows = []
    for config in CONFIGURATIONS:
        path = Path(f"/kaggle/working/{config['name']}_results.csv")
        if path.exists():
            rows.append(pd.read_csv(path))
    if rows:
        pd.concat(rows).to_csv("/kaggle/working/all_results.csv", index=False)
        print("\nFinal results saved to /kaggle/working/all_results.csv")

In [ ]:
import shutil

RESUME_DIR = Path("/kaggle/input/datasets/aondonamoses/soundspecific/checkpoints")
CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

for ckpt in RESUME_DIR.glob("*.pth"):
    dest = CHECKPOINT_DIR / ckpt.name
    shutil.copy(ckpt, dest)
    print(f"Restored: {ckpt.name}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
folds = get_cv_folds(COMBINED_DIR / "manifest.csv")

REMAINING_CONFIGURATIONS = [
    {"name": "resnet34_baseline",      "encoder": "resnet34",        "us_aug": False, "tta": False},
    {"name": "efficientnet_baseline",  "encoder": "efficientnet-b3", "us_aug": False, "tta": False},
    {"name": "efficientnet_tta",       "encoder": "efficientnet-b3", "us_aug": False, "tta": True},
    {"name": "efficientnet_usaug",     "encoder": "efficientnet-b3", "us_aug": True,  "tta": False},
    {"name": "efficientnet_usaug_tta", "encoder": "efficientnet-b3", "us_aug": True,  "tta": True},
]

all_results = {}
for config in REMAINING_CONFIGURATIONS:
    fold_results, summary = train_configuration(config, folds, device)
    all_results[config["name"]] = {"fold_results": fold_results, "summary": summary}
    config_row = {"configuration": config["name"]}
    for metric, vals in summary.items():
        config_row[f"{metric}_mean"] = round(vals["mean"], 4)
        config_row[f"{metric}_std"] = round(vals["std"], 4)
    pd.DataFrame([config_row]).to_csv(f"/kaggle/working/{config['name']}_results.csv", index=False)
    print(f"Saved: {config['name']}_results.csv")

Restored: resnet50_baseline_fold1.pth
Restored: resnet50_usaug_fold1.pth
Restored: resnet50_usaug_fold3.pth
Restored: resnet50_tta_fold4.pth
Restored: resnet50_baseline_fold2.pth
Restored: resnet50_baseline_fold4.pth
Restored: resnet50_baseline_fold0.pth
Restored: resnet34_baseline_fold1.pth
Restored: resnet50_tta_fold2.pth
Restored: resnet50_tta_fold3.pth
Restored: resnet34_baseline_fold4.pth
Restored: resnet50_tta_fold1.pth
Restored: resnet50_usaug_fold2.pth
Restored: resnet50_usaug_fold4.pth
Restored: resnet50_baseline_fold3.pth
Restored: resnet34_baseline_fold2.pth
Restored: resnet34_baseline_fold0.pth
Restored: resnet50_tta_fold0.pth
Restored: resnet50_usaug_fold0.pth
Restored: resnet34_baseline_fold3.pth

Configuration: resnet34_baseline

Fold 1/5


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
